# Analyzing Excel Data with Python, Pandas, and NumPy

Welcome to the integrated Data Analysis Lab! In real-world data science, engineering, and business workflows, raw data frequently lives inside spreadsheet files (`.xlsx`). Python empowers you to build fast, automated analytical pipelines by combining three core capabilities:

1. **File Handling:** Safely locating, reading, and exporting files to disk without manual human copy-pasting.
2. **Pandas:** Structuring tabular data into labeled DataFrames, computing business metrics, and querying subsets using label (`.loc`) and position (`.iloc`) selectors.
3. **NumPy:** Offloading high-performance numerical operations to vectorized C-level arrays (`ndarray`) for rapid statistical summarization, masking, and multi-dimensional matrix operations.

```text
┌─────────────────────────────────────────────────────────────────────────────┐
│                       The 2-Hour Analysis Pipeline                          │
│                                                                             │
│  [Disk File] ──► [File I/O Check] ──► [Pandas DataFrame] ──► [NumPy Engine] │
│   .xlsx file       os.path.exists()     Clean, Filter, Loc     Vectorized   │
│                                                                Calculations │
│                                                                     │       │
│  [Summary Report] ◄──── [File Export] ◄──── [Enriched Data] ◄───────┘       │
│   with open('...txt')    .to_excel()         Metrics & Profit               │
└─────────────────────────────────────────────────────────────────────────────┘
```

## Learning Objectives

By the end of this 2-hour lab, you will be able to:
- Verify file existence on disk using `os.path.exists()`.
- Ingest an Excel workbook directly into a Pandas `DataFrame` with `pd.read_excel()`.
- Inspect dimensions (`.shape`), data types (`.dtypes`), and compute summary statistics.
- Perform vectorized column arithmetic to generate new business KPIs (`Revenue`, `Profit`).
- Select rows and columns using positional indexer `.iloc[]` and label condition indexer `.loc[]`.
- Convert tabular series into high-speed NumPy `ndarray`s with `.to_numpy()`.
- Compute array-wide aggregations (`mean`, `median`, `std`, `min`, `max`, `percentile`).
- Apply conditional array transformations with `np.where()` without writing loops.
- Reshape 1D sales series into 2D matrices and compute axis-specific reductions (`axis=0`, `axis=1`).
- Export enriched datasets back to `.xlsx` using `.to_excel()` and generate an executive text report using Python's `with open(...)` file handler.

**Goal:** Build a complete end-to-end data analysis pipeline from raw Excel spreadsheet to automated analytical report.

In [ ]:
# Setup cell: installs openpyxl, imports libraries, and generates the sandbox Excel file.
import sys
import os
import numpy as np
import pandas as pd

# 1. Environment check
print("✅ Python Version:", sys.version.split()[0])
print("✅ Pandas Version:", pd.__version__)
print("✅ NumPy Version:", np.__version__)

# 2. Programmatically generate sandbox Excel file 'techgear_sales.xlsx'
excel_filename = "techgear_sales.xlsx"

records = [
    {"OrderID": "TG-1001", "Branch": "Tunis", "Category": "Laptops", "Product": "Pro Laptop 15", "Units_Sold": 25, "Unit_Price": 1200.0, "Cost_Per_Unit": 800.0, "Customer_Rating": 4.8},
    {"OrderID": "TG-1002", "Branch": "Sfax", "Category": "Audio", "Product": "Wireless Earbuds", "Units_Sold": 40, "Unit_Price": 75.0, "Cost_Per_Unit": 45.0, "Customer_Rating": 4.5},
    {"OrderID": "TG-1003", "Branch": "Sousse", "Category": "Monitors", "Product": "27-inch 4K Monitor", "Units_Sold": 15, "Unit_Price": 350.0, "Cost_Per_Unit": 220.0, "Customer_Rating": 4.7},
    {"OrderID": "TG-1004", "Branch": "Ariana", "Category": "Accessories", "Product": "Ergonomic Mouse", "Units_Sold": 50, "Unit_Price": 35.0, "Cost_Per_Unit": 18.0, "Customer_Rating": 4.3},
    {"OrderID": "TG-1005", "Branch": "Tunis", "Category": "Audio", "Product": "Noise-Canceling Headphones", "Units_Sold": 30, "Unit_Price": 180.0, "Cost_Per_Unit": 110.0, "Customer_Rating": 4.9},
    {"OrderID": "TG-1006", "Branch": "Sfax", "Category": "Laptops", "Product": "Ultra Laptop 13", "Units_Sold": 20, "Unit_Price": 950.0, "Cost_Per_Unit": 650.0, "Customer_Rating": 4.6},
    {"OrderID": "TG-1007", "Branch": "Sousse", "Category": "Accessories", "Product": "Mechanical Keyboard", "Units_Sold": 35, "Unit_Price": 85.0, "Cost_Per_Unit": 50.0, "Customer_Rating": 4.4},
    {"OrderID": "TG-1008", "Branch": "Ariana", "Category": "Monitors", "Product": "34-inch Ultrawide Monitor", "Units_Sold": 12, "Unit_Price": 520.0, "Cost_Per_Unit": 340.0, "Customer_Rating": 4.7},
    {"OrderID": "TG-1009", "Branch": "Tunis", "Category": "Accessories", "Product": "Ergonomic Mouse", "Units_Sold": 60, "Unit_Price": 35.0, "Cost_Per_Unit": 18.0, "Customer_Rating": 4.2},
    {"OrderID": "TG-1010", "Branch": "Sfax", "Category": "Monitors", "Product": "27-inch 4K Monitor", "Units_Sold": 18, "Unit_Price": 350.0, "Cost_Per_Unit": 220.0, "Customer_Rating": 4.6},
    {"OrderID": "TG-1011", "Branch": "Sousse", "Category": "Laptops", "Product": "Pro Laptop 15", "Units_Sold": 15, "Unit_Price": 1200.0, "Cost_Per_Unit": 800.0, "Customer_Rating": 4.8},
    {"OrderID": "TG-1012", "Branch": "Ariana", "Category": "Audio", "Product": "Noise-Canceling Headphones", "Units_Sold": 22, "Unit_Price": 180.0, "Cost_Per_Unit": 110.0, "Customer_Rating": 4.5},
    {"OrderID": "TG-1013", "Branch": "Tunis", "Category": "Monitors", "Product": "34-inch Ultrawide Monitor", "Units_Sold": 10, "Unit_Price": 520.0, "Cost_Per_Unit": 340.0, "Customer_Rating": 4.9},
    {"OrderID": "TG-1014", "Branch": "Sfax", "Category": "Accessories", "Product": "Mechanical Keyboard", "Units_Sold": 45, "Unit_Price": 85.0, "Cost_Per_Unit": 50.0, "Customer_Rating": 4.3},
    {"OrderID": "TG-1015", "Branch": "Sousse", "Category": "Audio", "Product": "Wireless Earbuds", "Units_Sold": 55, "Unit_Price": 75.0, "Cost_Per_Unit": 45.0, "Customer_Rating": 4.6},
    {"OrderID": "TG-1016", "Branch": "Ariana", "Category": "Laptops", "Product": "Ultra Laptop 13", "Units_Sold": 16, "Unit_Price": 950.0, "Cost_Per_Unit": 650.0, "Customer_Rating": 4.4},
    {"OrderID": "TG-1017", "Branch": "Tunis", "Category": "Laptops", "Product": "Ultra Laptop 13", "Units_Sold": 22, "Unit_Price": 950.0, "Cost_Per_Unit": 650.0, "Customer_Rating": 4.7},
    {"OrderID": "TG-1018", "Branch": "Sfax", "Category": "Laptops", "Product": "Pro Laptop 15", "Units_Sold": 14, "Unit_Price": 1200.0, "Cost_Per_Unit": 800.0, "Customer_Rating": 4.5},
    {"OrderID": "TG-1019", "Branch": "Sousse", "Category": "Monitors", "Product": "34-inch Ultrawide Monitor", "Units_Sold": 8, "Unit_Price": 520.0, "Cost_Per_Unit": 340.0, "Customer_Rating": 4.6},
    {"OrderID": "TG-1020", "Branch": "Ariana", "Category": "Accessories", "Product": "Ergonomic Mouse", "Units_Sold": 48, "Unit_Price": 35.0, "Cost_Per_Unit": 18.0, "Customer_Rating": 4.1},
    {"OrderID": "TG-1021", "Branch": "Tunis", "Category": "Audio", "Product": "Wireless Earbuds", "Units_Sold": 65, "Unit_Price": 75.0, "Cost_Per_Unit": 45.0, "Customer_Rating": 4.8},
    {"OrderID": "TG-1022", "Branch": "Sfax", "Category": "Audio", "Product": "Noise-Canceling Headphones", "Units_Sold": 28, "Unit_Price": 180.0, "Cost_Per_Unit": 110.0, "Customer_Rating": 4.7},
    {"OrderID": "TG-1023", "Branch": "Sousse", "Category": "Accessories", "Product": "Ergonomic Mouse", "Units_Sold": 42, "Unit_Price": 35.0, "Cost_Per_Unit": 18.0, "Customer_Rating": 4.2},
    {"OrderID": "TG-1024", "Branch": "Ariana", "Category": "Monitors", "Product": "27-inch 4K Monitor", "Units_Sold": 20, "Unit_Price": 350.0, "Cost_Per_Unit": 220.0, "Customer_Rating": 4.6}
]

df_setup = pd.DataFrame(records)
df_setup.to_excel(excel_filename, index=False)
print(f"✅ Setup complete! Verified '{excel_filename}' ({df_setup.shape[0]} rows, {df_setup.shape[1]} columns).")

## Part 1 — Guided Mechanics: File Ingestion & Pandas Exploration

### 1.1 Verifying File Existence on Disk

Before reading files in Python, it is standard defensive programming practice to verify that the target file exists. The built-in `os.path.exists(path)` function returns `True` if the path exists on disk, and `False` otherwise.

```text
               os.path.exists("techgear_sales.xlsx")
                             │
            ┌────────────────┴────────────────┐
            ▼                                 ▼
      File is found                     File missing
      Returns: True                    Returns: False
```

**Goal:** Check whether `"techgear_sales.xlsx"` exists using `os.path.exists()`, store the result in `file_exists`, and print the result.

**Expected output:**
```text
File 'techgear_sales.xlsx' exists: True
```

In [ ]:
file_name = "techgear_sales.xlsx"

# TODO: Use os.path.exists() to check if file_name exists on disk
file_exists = ...

print(f"File '{file_name}' exists:", file_exists)

# Verification
assert file_exists is True, f"Expected {file_name} to exist, but got False!"
print("✅ Test passed!")

### 1.2 Loading Excel Spreadsheets with Pandas

Pandas provides `pd.read_excel()` to load Microsoft Excel spreadsheets (`.xlsx`) directly into memory as a two-dimensional labeled tabular `DataFrame`.

```text
Excel File on Disk:  techgear_sales.xlsx
                             │
                      pd.read_excel(...)
                             ▼
In-Memory DataFrame:  df (24 rows × 8 columns)
```

**Goal:** Load `"techgear_sales.xlsx"` into a variable named `df` using `pd.read_excel()`. Display the first 5 rows using `df.head()`.

**Expected output:**
A DataFrame preview showing the first 5 rows and 8 columns (`OrderID`, `Branch`, `Category`, `Product`, `Units_Sold`, `Unit_Price`, `Cost_Per_Unit`, `Customer_Rating`).

In [ ]:
# TODO: Read 'techgear_sales.xlsx' into a DataFrame named df using pd.read_excel()
df = ...

print("First 5 rows of TechGear Sales:")
display(df.head())

# Verification
assert isinstance(df, pd.DataFrame), "df should be a pandas DataFrame!"
assert len(df) == 24, f"Expected 24 rows, but got {len(df)}!"
print("✅ Test passed! DataFrame successfully loaded.")

### 1.3 Inspecting DataFrame Dimensions and Types

A DataFrame provides structural metadata attributes:
- `.shape`: A tuple `(rows, columns)` indicating dimensions.
- `.dtypes`: The data type of each column (`int64`, `float64`, `object`).
- `.isna().sum()`: Checks for any missing values across each column.

```text
df.shape  ──►  (24, 8)
                │   │
             rows  columns
```

**Goal:** Extract the row count and column count from `df.shape` into `num_rows` and `num_cols`. Check that there are zero missing values in total across the entire DataFrame.

**Expected output:**
```text
Rows: 24, Columns: 8
Total missing values: 0
```

In [ ]:
# TODO 1: Extract row count (index 0 of df.shape)
num_rows = ...

# TODO 2: Extract column count (index 1 of df.shape)
num_cols = ...

# TODO 3: Compute the grand total of missing values across all columns
total_missing = ...

print(f"Rows: {num_rows}, Columns: {num_cols}")
print(f"Total missing values: {total_missing}")

# Verification
assert num_rows == 24 and num_cols == 8, f"Expected (24, 8), got ({num_rows}, {num_cols})"
assert total_missing == 0, f"Expected 0 missing values, got {total_missing}"
print("✅ Test passed! Dataset dimensions and cleanliness verified.")

### 1.4 Feature Engineering with Vectorized Column Arithmetic

In Pandas, arithmetic between columns operates element-by-element across every row simultaneously without writing a `for` loop.

```text
df['Revenue'] = df['Units_Sold'] * df['Unit_Price']
df['Profit']  = df['Revenue'] - (df['Units_Sold'] * df['Cost_Per_Unit'])
```

**Goal:**
1. Create a new column `'Revenue'` in `df` representing `Units_Sold * Unit_Price`.
2. Create a new column `'Profit'` in `df` representing `Revenue - (Units_Sold * Cost_Per_Unit)`.
3. Calculate the sum of total revenue (`total_revenue`) and total profit (`total_profit`).

**Expected output:**
```text
Total Revenue: $194,250.00
Total Profit: $67,890.00
```

In [ ]:
# TODO 1: Calculate 'Revenue' as Units_Sold multiplied by Unit_Price
df['Revenue'] = ...

# TODO 2: Calculate 'Profit' as Revenue minus (Units_Sold multiplied by Cost_Per_Unit)
df['Profit'] = ...

# TODO 3: Sum total revenue and total profit
total_revenue = ...
total_profit = ...

print(f"Total Revenue: ${total_revenue:,.2f}")
print(f"Total Profit: ${total_profit:,.2f}")

# Verification
assert total_revenue == 194250.0, f"Expected 194250.0, got {total_revenue}"
assert total_profit == 67890.0, f"Expected 67890.0, got {total_profit}"
print("✅ Test passed! Financial metrics successfully calculated.")

### 1.5 Positional (`.iloc`) vs. Label (`.loc`) Indexing

Pandas provides two explicit indexers to extract subsets:
- `.iloc[row_positions, col_positions]`: Integer position-based indexing (0-indexed).
- `.loc[row_conditions, column_labels]`: Label and boolean condition-based indexing.

```text
.iloc[0:3, [0, 3, 8]]    ──►  First 3 rows, specific column integer positions
.loc[df['Branch'] == 'Tunis', ['OrderID', 'Product', 'Revenue']]  ──► Filter by condition & labels
```

**Goal:**
1. Use `.iloc` to slice the first 3 rows (`0:3`) from `df[cols_to_view]`. Store in `top3_orders`.
2. Use `.loc` to select all transactions from the `'Tunis'` branch for columns `['OrderID', 'Product', 'Revenue']`. Store in `tunis_orders`.

**Expected output:**
`top3_orders` will contain 3 rows.
`tunis_orders` will contain 6 rows.

In [ ]:
cols_to_view = ['OrderID', 'Product', 'Revenue']

# TODO 1: Use .iloc to get the first 3 rows (indices 0:3) of cols_to_view
top3_orders = df[cols_to_view].iloc[...]

# TODO 2: Use .loc to get rows where Branch == 'Tunis' with cols_to_view
tunis_orders = df.loc[..., cols_to_view]

print("Top 3 Orders (.iloc):")
display(top3_orders)

print("Tunis Branch Orders (.loc):")
display(tunis_orders)

# Verification
assert len(top3_orders) == 3, f"Expected 3 rows in top3_orders, got {len(top3_orders)}"
assert len(tunis_orders) == 6, f"Expected 6 Tunis orders, got {len(tunis_orders)}"
print("✅ Test passed! Indexing with .iloc and .loc mastered.")

## Part 2 — Guided Mechanics: NumPy Vectorization & Matrix Math

### 2.1 Bridging Pandas Series to Pure NumPy Arrays

While Pandas is ideal for structured tables, NumPy provides the optimized numerical engine under the hood. You can extract any column into a pure contiguous NumPy `ndarray` using `.to_numpy()`.

```text
Pandas Series:  df['Revenue'] (labeled index + values)
                     │
               .to_numpy()
                     ▼
NumPy Array:    array([30000., 3000., 5250., ...]) (contiguous C-array)
Type:           numpy.ndarray
Shape:          (24,)
```

**Goal:** Extract the `'Revenue'` column as a NumPy array named `revenue_arr`, and `'Units_Sold'` as `units_arr`. Print their types and shapes.

**Expected output:**
```text
Revenue array type: <class 'numpy.ndarray'>, shape: (24,)
Units array type: <class 'numpy.ndarray'>, shape: (24,)
```

In [ ]:
# TODO 1: Extract df['Revenue'] to a NumPy array named revenue_arr
revenue_arr = ...

# TODO 2: Extract df['Units_Sold'] to a NumPy array named units_arr
units_arr = ...

print(f"Revenue array type: {type(revenue_arr)}, shape: {revenue_arr.shape}")
print(f"Units array type: {type(units_arr)}, shape: {units_arr.shape}")

# Verification
assert isinstance(revenue_arr, np.ndarray), "revenue_arr must be a numpy.ndarray!"
assert isinstance(units_arr, np.ndarray), "units_arr must be a numpy.ndarray!"
assert revenue_arr.shape == (24,), f"Expected shape (24,), got {revenue_arr.shape}"
print("✅ Test passed! Columns converted to NumPy arrays.")

### 2.2 Statistical Benchmarks with NumPy

NumPy provides highly optimized statistical calculation functions that process entire arrays in compiled C:
- `np.mean(a)`: Arithmetic mean (average).
- `np.median(a)`: Median value (50th percentile).
- `np.std(a)`: Standard deviation.
- `np.min(a)`, `np.max(a)`: Minimum and maximum values.
- `np.percentile(a, q)`: The $q$-th percentile.

```text
revenue_arr ──► np.mean()       ──► 8093.75
            ──► np.median()     ──► 5120.00
            ──► np.percentile() ──► 9050.00 (75th percentile)
```

**Goal:** Compute the following metrics on `revenue_arr`:
1. `mean_rev`: average revenue rounded to 2 decimal places.
2. `median_rev`: median revenue.
3. `p75_rev`: the 75th percentile of revenue.
4. `max_rev` and `min_rev`: highest and lowest revenue.

**Expected output:**
```text
Mean Revenue: $8,093.75
Median Revenue: $5,120.00
75th Percentile Revenue: $9,050.00
Max: $30,000.00, Min: $1,470.00
```

In [ ]:
# TODO 1: Compute mean revenue using np.mean()
mean_rev = ...

# TODO 2: Compute median revenue using np.median()
median_rev = ...

# TODO 3: Compute 75th percentile of revenue using np.percentile()
p75_rev = ...

# TODO 4: Compute max and min revenue using np.max() and np.min()
max_rev = ...
min_rev = ...

print(f"Mean Revenue: ${mean_rev:,.2f}")
print(f"Median Revenue: ${median_rev:,.2f}")
print(f"75th Percentile Revenue: ${p75_rev:,.2f}")
print(f"Max: ${max_rev:,.2f}, Min: ${min_rev:,.2f}")

# Verification
assert np.isclose(mean_rev, 8093.75), f"Expected 8093.75, got {mean_rev}"
assert np.isclose(median_rev, 5120.0), f"Expected 5120.0, got {median_rev}"
assert np.isclose(p75_rev, 9050.0), f"Expected 9050.0, got {p75_rev}"
print("✅ Test passed! Statistical aggregations verified.")

### 2.3 Vectorized Conditional Logic with `np.where()`

In Python, setting values based on conditions across thousands of items with `if/else` loops is slow. NumPy's `np.where(condition, value_if_true, value_if_false)` vectorizes this logic across the entire array at native C speeds.

```text
                 np.where(revenue_arr >= 10000, "High", "Standard")
                                        │
                 ┌──────────────────────┴──────────────────────┐
                 ▼                                             ▼
            Revenue >= 10000                              Revenue < 10000
           Assign: "High"                                Assign: "Standard"
```

**Goal:**
1. Use `np.where()` to assign transactions with revenue $\ge 10,000$ to `"High"`, and all others to `"Standard"`. Store this in `tier_arr`.
2. Add `tier_arr` back to the DataFrame as a new column named `'Sales_Tier'`.
3. Count how many transactions are classified as `"High"` and `"Standard"`.

**Expected output:**
```text
High tier orders: 6
Standard tier orders: 18
```

In [ ]:
# TODO 1: Use np.where to assign "High" if revenue_arr >= 10000, else "Standard"
tier_arr = np.where(..., "High", "Standard")

# TODO 2: Assign tier_arr to df['Sales_Tier']
df['Sales_Tier'] = ...

# TODO 3: Count occurrences using sum on boolean comparison
high_count = int(np.sum(tier_arr == "High"))
standard_count = int(np.sum(tier_arr == "Standard"))

print(f"High tier orders: {high_count}")
print(f"Standard tier orders: {standard_count}")

# Verification
assert high_count == 6, f"Expected 6 High orders, got {high_count}"
assert standard_count == 18, f"Expected 18 Standard orders, got {standard_count}"
print("✅ Test passed! Vectorized conditional assignment complete.")

### 2.4 Reshaping and 2D Matrix Reductions

NumPy excels at multidimensional matrix algebra. With `.reshape(rows, cols)`, flat 1D arrays are transformed into 2D grids without duplicating memory.

When reducing along an axis:
- `axis=0`: Operates downwards across rows (producing column-wise summaries).
- `axis=1`: Operates across columns (producing row-wise summaries).

```text
4x4 Unit Sales Grid:
                 col 0   col 1   col 2   col 3
        row 0 ┌   25      40      15      50   ┐ ──► row sum (axis=1): 130
        row 1 │   30      20      35      12   │ ──► row sum (axis=1): 97
        row 2 │   60      18      15      22   │ ──► row sum (axis=1): 115
        row 3 └   10      45      55      16   ┘ ──► row sum (axis=1): 126
                   │       │       │       │
col mean (axis=0):31.25   30.75   30.0    25.0
```

**Goal:**
1. Take the first 16 values of `units_arr` and reshape them into a 4x4 matrix named `matrix_4x4`.
2. Compute the sum of each row across columns using `matrix_4x4.sum(axis=1)`. Store in `branch_totals`.
3. Compute the mean of each column across rows using `matrix_4x4.mean(axis=0)`. Store in `product_averages`.
4. Slice and extract the center 2x2 subgrid (`rows 1:3, cols 1:3`).

**Expected output:**
```text
4x4 Matrix:
[[25 40 15 50]
 [30 20 35 12]
 [60 18 15 22]
 [10 45 55 16]]
Row totals (axis=1): [130  97 115 126]
Column averages (axis=0): [31.25 30.75 30.   25.  ]
Center 2x2:
[[20 35]
 [18 15]]
```

In [ ]:
# TODO 1: Slice first 16 items of units_arr and reshape into (4, 4)
matrix_4x4 = units_arr[:16].reshape(...)

# TODO 2: Compute row totals across columns using axis=1
branch_totals = matrix_4x4.sum(axis=...)

# TODO 3: Compute column averages across rows using axis=0
product_averages = matrix_4x4.mean(axis=...)

# TODO 4: Slice the center 2x2 subgrid (rows 1:3, columns 1:3)
center_subgrid = matrix_4x4[..., ...]

print("4x4 Matrix:")
print(matrix_4x4)
print("Row totals (axis=1):", branch_totals)
print("Column averages (axis=0):", product_averages)
print("Center 2x2:")
print(center_subgrid)

# Verification
assert matrix_4x4.shape == (4, 4), f"Expected (4, 4), got {matrix_4x4.shape}"
assert np.array_equal(branch_totals, np.array([130, 97, 115, 126])), "Row totals mismatch!"
assert np.array_equal(center_subgrid, np.array([[20, 35], [18, 15]])), "Center subgrid mismatch!"
print("✅ Test passed! 2D matrix operations and axis reductions verified.")

## Part 3 — Solo Application: Business Performance Pipeline

### 3.1 Multi-Condition Filtering in Pandas

In business reporting, analysts frequently isolate transactions that satisfy multiple criteria. In Pandas, combine conditions using the bitwise AND operator `&`, wrapping each condition in parentheses:

```text
df[(df['Branch'] == 'Tunis') & (df['Revenue'] >= 5000)]
```

**Goal:**
1. Filter `df` to isolate orders from branch `'Tunis'` that generated at least `$5,000` in `Revenue`. Store in `tunis_high_revenue`.
2. Display the filtered results and count the matching orders.

**Expected output:**
```text
Found 4 high-revenue orders in Tunis.
```

In [ ]:
# TODO 1: Filter df for Branch == 'Tunis' AND Revenue >= 5000
tunis_high_revenue = df[(...) & (...)]

# TODO 2: Count the matching rows
count_tunis_high = len(tunis_high_revenue)

print(f"Found {count_tunis_high} high-revenue orders in Tunis.")
display(tunis_high_revenue[['OrderID', 'Branch', 'Product', 'Revenue', 'Sales_Tier']])

# Verification
assert count_tunis_high == 4, f"Expected 4 orders, got {count_tunis_high}"
print("✅ Test passed! Multi-condition filtering successful.")

### 3.2 Vectorized Profit Margin & Array Boolean Masking

The profit margin ratio measures profitability per dollar earned:
$$\text{Margin Ratio} = \frac{\text{Profit}}{\text{Revenue}}$$

Using NumPy array element-wise division, we can compute this ratio across all orders in one operation, calculate the mean profit margin, and apply a boolean mask to isolate orders beating the average.

```text
margin_ratios = profit_arr / revenue_arr
high_margin_mask = margin_ratios > mean_margin
```

**Goal:**
1. Extract `profit_arr = df['Profit'].to_numpy()`.
2. Compute `margin_ratios = profit_arr / revenue_arr`.
3. Calculate `mean_margin` using `np.mean()`.
4. Create a boolean mask `above_mean_mask` identifying orders where `margin_ratios > mean_margin`.
5. Count how many orders exceeded the mean margin ratio using `.sum()`.

**Expected output:**
```text
Mean Profit Margin: 38.47%
Orders exceeding mean margin: 12
```

In [ ]:
# TODO 1: Extract Profit to a numpy array
profit_arr = ...

# TODO 2: Compute margin ratios by dividing profit_arr by revenue_arr
margin_ratios = ...

# TODO 3: Compute mean profit margin using np.mean()
mean_margin = ...

# TODO 4: Create boolean mask for orders where margin_ratios > mean_margin
above_mean_mask = ...

# TODO 5: Count how many orders are True in the mask
above_mean_count = ...

print(f"Mean Profit Margin: {mean_margin * 100:.2f}%")
print(f"Orders exceeding mean margin: {above_mean_count}")

# Verification
assert np.isclose(mean_margin, 0.3847, atol=1e-3), f"Expected ~0.3847, got {mean_margin}"
assert above_mean_count == 12, f"Expected 12 orders, got {above_mean_count}"
print("✅ Test passed! Profit margin vectorization and masking verified.")

### 3.3 Exporting Enriched Data back to Excel

After processing and enriching the data in Pandas and NumPy, data pipelines must persist results to disk for stakeholders. Pandas provides `df.to_excel(filename, index=False)` to write DataFrames directly to `.xlsx`.

```text
In-Memory Enriched DataFrame ──► df.to_excel("enriched_techgear_sales.xlsx", index=False)
                                                        │
                                                        ▼
                                       New file on disk ready for Excel
```

**Goal:**
1. Add a rounded `'Profit_Margin'` column to `df` (`np.round(margin_ratios, 4)`).
2. Export `df` to `'enriched_techgear_sales.xlsx'` with `index=False`.
3. Use `os.path.exists()` to verify the file was created.

**Expected output:**
```text
File 'enriched_techgear_sales.xlsx' created successfully: True
Exported DataFrame shape: (24, 12)
```

In [ ]:
export_excel_file = "enriched_techgear_sales.xlsx"

# TODO 1: Add 'Profit_Margin' column to df (round margin_ratios to 4 decimal places)
df['Profit_Margin'] = ...

# TODO 2: Export df to export_excel_file without index using .to_excel()
df.to_excel(..., index=False)

# TODO 3: Verify that the exported file exists using os.path.exists()
export_exists = os.path.exists(...)

print(f"File '{export_excel_file}' created successfully: {export_exists}")
print(f"Exported DataFrame shape: {df.shape}")

# Verification
assert export_exists is True, f"File {export_excel_file} was not found on disk!"
assert df.shape == (24, 12), f"Expected (24, 12), got {df.shape}"
print("✅ Test passed! Enriched dataset persisted to Excel.")

### 3.4 Automated Executive Summary File with Python File I/O

In addition to spreadsheet files, automated pipelines often generate plain text executive summaries or log files for management.

Using Python's built-in `with open(filename, 'w') as f:` context manager guarantees the file handle is properly closed when finished:

```text
with open("executive_summary.txt", "w") as f:
    f.write("...
")
```

**Goal:**
1. Compute the top-selling branch by total revenue using `df.groupby('Branch')['Revenue'].sum().idxmax()`.
2. Write a summary text file named `"executive_summary.txt"` with the exact lines:
   - Line 1: `TECHGEAR SALES EXECUTIVE SUMMARY`
   - Line 2: `Total Orders: 24`
   - Line 3: `Total Revenue: $194,250.00`
   - Line 4: `Total Profit: $67,890.00`
   - Line 5: `Top Branch: Tunis`
3. Read the file back line-by-line using `with open("executive_summary.txt", "r") as f:` and print each line stripped of extra whitespace.

**Expected output:**
```text
--- EXECUTIVE SUMMARY REPORT ---
TECHGEAR SALES EXECUTIVE SUMMARY
Total Orders: 24
Total Revenue: $194,250.00
Total Profit: $67,890.00
Top Branch: Tunis
```

In [ ]:
summary_file = "executive_summary.txt"

# TODO 1: Identify the top branch by revenue
top_branch = df.groupby('Branch')['Revenue'].sum().idxmax()

# TODO 2: Open summary_file in write mode ('w') and write the 5 lines
with open(summary_file, '___') as f:
    f.write("TECHGEAR SALES EXECUTIVE SUMMARY\n")
    f.write(f"Total Orders: {len(df)}\n")
    f.write(f"Total Revenue: ${total_revenue:,.2f}\n")
    f.write(f"Total Profit: ${total_profit:,.2f}\n")
    f.write(f"Top Branch: {top_branch}\n")

# TODO 3: Read back and print the file line by line in read mode ('r')
print("--- EXECUTIVE SUMMARY REPORT ---")
with open(summary_file, '___') as f:
    for line in f:
        print(line.strip())

# Verification
assert os.path.exists(summary_file), f"File {summary_file} does not exist!"
with open(summary_file, "r") as f:
    lines = [line.strip() for line in f]
assert len(lines) == 5, f"Expected 5 lines, got {len(lines)}"
assert lines[4] == "Top Branch: Tunis", f"Expected 'Top Branch: Tunis', got '{lines[4]}'"
print("✅ Test passed! Automated text report created and verified.")

## Part 4 — The Bug Hunt

### 4.1 Debugging Challenge: The Missing Bridge between Pandas and NumPy

A common source of errors for beginners is attempting to call NumPy array methods (such as `.reshape()`) directly on a Pandas `Series` without converting it first.

Run the cell below as-is to inspect the deliberate traceback:

```text
AttributeError: 'Series' object has no attribute 'reshape'
```

**Task:**
1. Run the buggy code cell to observe the traceback.
2. Read the error message carefully: why does a Pandas Series not have a `.reshape` attribute?
3. Fix the bug by converting the Series to a NumPy array using `.to_numpy()` before slicing and reshaping.

**Goal:** Correctly reshape the first 12 units sold into a 3x4 matrix without triggering an `AttributeError`.

**Expected output:**
```text
Reshaped 3x4 units matrix:
[[25 40 15 50]
 [30 20 35 12]
 [60 18 15 22]]
```

In [ ]:
# BUG HUNT: Run this cell to observe the AttributeError, then fix it!
# Hint: df['Units_Sold'] is a Pandas Series. Call .to_numpy() before slicing and reshaping.

# BUGGY LINE:
# reshaped_units = df['Units_Sold'][:12].reshape(3, 4)

# TODO: Fix the line below by converting to numpy array first
reshaped_units = df['Units_Sold']...[:12].reshape(3, 4)

print("Reshaped 3x4 units matrix:")
print(reshaped_units)

# Verification
assert isinstance(reshaped_units, np.ndarray), "reshaped_units must be a NumPy ndarray!"
assert reshaped_units.shape == (3, 4), f"Expected shape (3, 4), got {reshaped_units.shape}"
print("✅ Bug successfully identified and fixed!")

## Part 5 — Scientific Reflection

Answer the following conceptual questions to solidify your understanding of how File Handling, Pandas, and NumPy work together:

### Question 1: Data Structures and Memory
Why does a Pandas Series carry an `index` while a NumPy `ndarray` only has integer positions? In what analytical situations is the Pandas labeled index advantageous, and when is a pure NumPy array more efficient?

> **Your Answer:**
> _Type your answer here..._

---

### Question 2: Vectorization vs. Standard Python Loops
Compare computing `df['Revenue'] = df['Units_Sold'] * df['Unit_Price']` against iterating through rows with a Python `for` loop. What happens behind the scenes in compiled C memory during vectorized operations?

> **Your Answer:**
> _Type your answer here..._

---

### Question 3: The Persistence Lifecycle
Explain the difference between working with data in variables (`df`, `revenue_arr`) versus saving to disk (`techgear_sales.xlsx`, `executive_summary.txt`). What occurs to in-memory data when the Python kernel restarts, and why does automated file output complete the analytical pipeline?

> **Your Answer:**
> _Type your answer here..._